[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **AI-Powered Cricket Bowling Biomechanics Analyzer with Real-Time Motion Tracking**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Project Overview

An advanced cricket biomechanics analysis system that leverages fine-tuned **YOLOv8x-Pose** keypoint detection architecture to transform raw broadcast footage into actionable bowling intelligence. Unlike standard pose estimation projects, it processes bowler movement data into a **Biomechanical Motion Schematic**, providing real-time arm trajectory visualization and elbow hyperextension analysis.

The system performs three core functions simultaneously:
1. **3-Point Keypoint Detection:** High-precision localization of Shoulder, Elbow, and Wrist joints from live video frames.
2. **Dynamic Arm Trajectory Mapping:** A custom motion trail system that tracks the wrist arc across frames, visualizing the full bowling arm swing with a color-coded gradient overlay.
3. **Biomechanical Analytics HUD:** A real-time side panel displaying elbow angle and wrist speed graphs, mirroring professional-grade sports analytics dashboards.

---



## Real-World Applications

### 1. Illegal Bowling Action Detection
The system monitors elbow bend angle throughout the delivery stride. By tracking hyperextension beyond the ICC-mandated threshold, it can flag potentially illegal actions for umpire review replacing costly biomechanics lab sessions with real-time broadcast analysis.

### 2. Player Performance Analytics
The wrist speed and arm trajectory data can be integrated into professional coaching dashboards (like those used by BCCI, Cricket Australia, or ECB), allowing coaches to compare delivery mechanics across multiple bowlers and identify technique inefficiencies.

### 3. Broadcast Enhancement
The motion trail and HUD overlay can be applied live to broadcast feeds, providing viewers with the same depth of biomechanical insight currently only available in post-match analysis segments - similar to Hawk-Eye or Ultra-Edge overlays.

### 4. Injury Prevention & Rehabilitation
The elbow angle time-series data can be monitored across training sessions to detect gradual changes in bowling mechanics that may indicate early signs of stress injury, enabling proactive intervention before career-threatening damage occurs.

##  Installation & Setup

To use the **Labellerr Fine-Tune Utilities** in your environment (Kaggle, Colab, or Local), you can clone the repository directly into your workspace. This allows you to access the specialized conversion scripts for your ADAS dataset.

In [4]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


## Random Frame Extraction from Video

Extracts a fixed number of high-quality frames from one or more videos to create an image dataset for annotation and training.

### 🔹 Purpose
- Convert raw manufacturing videos into individual image frames  
- Perform random sampling to avoid frame bias  
- Prepare data for annotation and YOLO training  


In [ ]:
from yolo_finetune_utils.frame_extractor import extract_random_frames

extract_random_frames(
    paths=['clip.mp4'],
    total_images=100,
    out_dir="input_data",
    jpg_quality=100,
    seed=42
)

[✓] Extracted 100 frames to folder: input_data


## Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


## COCO JSON → YOLO Pose Label Converter

A utility script that converts COCO keypoint annotation format into YOLO Pose `.txt` label files for training.

### What It Does
- Reads a COCO JSON file containing keypoint annotations
- Normalizes bounding box coordinates and 3-point keypoints (Shoulder, Elbow, Wrist) relative to image dimensions
- Outputs one `.txt` label file per image in YOLO Pose format

### Output Format
```
class cx cy bw bh  kp1x kp1y vis1  kp2x kp2y vis2  kp3x kp3y vis3
```

### Usage
1. Set `COCO_PATH` to your COCO JSON file
2. Set `OUTPUT_DIR` to desired labels folder
3. Run — labels are saved and a sample is printed for verification

In [2]:
import json
import os

# ---- CONFIG ----
COCO_PATH = "annotations/coco_annotations.json"
OUTPUT_DIR = "labels_all"
NUM_KP = 3
# ----------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(COCO_PATH, 'r') as f:
    coco = json.load(f)

# Build image id -> image info map
image_map = {img['id']: img for img in coco['images']}

saved = 0

for ann in coco['annotations']:
    img_info = image_map[ann['image_id']]
    img_w = img_info['width']
    img_h = img_info['height']
    stem  = os.path.splitext(img_info['file_name'])[0]

    # Bounding box normalized
    x, y, bw, bh = ann['bbox']
    cx = (x + bw / 2) / img_w
    cy = (y + bh / 2) / img_h
    nw = bw / img_w
    nh = bh / img_h

    # Keypoints normalized
    kps = ann['keypoints']
    kp_str = ""
    for i in range(NUM_KP):
        kx  = kps[i*3]     / img_w
        ky  = kps[i*3 + 1] / img_h
        vis = kps[i*3 + 2]
        kp_str += f" {kx:.6f} {ky:.6f} {vis}"

    line = f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}{kp_str}"

    with open(os.path.join(OUTPUT_DIR, stem + ".txt"), 'w') as f:
        f.write(line)

    saved += 1

print(f"Done! {saved} YOLO label files saved to {OUTPUT_DIR}/")

# Verify sample
sample = os.listdir(OUTPUT_DIR)[0]
with open(os.path.join(OUTPUT_DIR, sample)) as f:
    print(f"\nSample ({sample}):")
    print(f.read())

Done! 96 YOLO label files saved to labels_all/

Sample (frame_0000.txt):
0 0.060690 0.223185 0.056474 0.149981 0.042870 0.279657 2 0.078510 0.246602 2 0.074641 0.166713 2


## Train/Val Dataset Splitter

A utility script that splits labeled frames into training and validation sets for YOLO Pose model training.

### What It Does
- Reads all label files from `labels_all/` and matches them with frames
- Randomly shuffles and splits into **85 train / 11 val** (configurable)
- Creates the complete `dataset/` folder structure and copies images + labels

### Output Structure
```
dataset/
├── train/
│   ├── images/   ← 85 frames
│   └── labels/   ← 85 YOLO label files
└── val/
    ├── images/   ← 11 frames
    └── labels/   ← 11 YOLO label files
```

### Usage
1. Set `FRAMES_FOLDER` and `LABELS_FOLDER` paths
2. Adjust `TRAIN_COUNT` and `VAL_COUNT` as needed
3. Run — split summary is printed on completion

In [4]:
import os
import shutil
import random

# ---- CONFIG ----
FRAMES_FOLDER = "frames"
LABELS_FOLDER = "labels_all"
TRAIN_COUNT = 85
VAL_COUNT = 15
RANDOM_SEED = 42
# ----------------

random.seed(RANDOM_SEED)

# Get only frames that have a matching label (processed ones)
label_files = os.listdir(LABELS_FOLDER)
stems = [os.path.splitext(f)[0] for f in label_files]  # frame_0000, frame_0001 etc

print(f"Total labeled frames: {len(stems)}")

# Shuffle and split
random.shuffle(stems)
train_stems = stems[:TRAIN_COUNT]
val_stems   = stems[TRAIN_COUNT:TRAIN_COUNT + VAL_COUNT]

print(f"Train: {len(train_stems)} | Val: {len(val_stems)}")

# Create folders
for folder in [
    "dataset/train/images",
    "dataset/train/labels",
    "dataset/val/images",
    "dataset/val/labels"
]:
    os.makedirs(folder, exist_ok=True)

# Copy files
def copy_split(stems, split):
    missing_imgs = 0
    for stem in stems:
        # Copy image
        src_img = os.path.join(FRAMES_FOLDER, stem + ".jpg")
        dst_img = os.path.join(f"dataset/{split}/images", stem + ".jpg")
        if os.path.exists(src_img):
            shutil.copy(src_img, dst_img)
        else:
            print(f"WARNING image not found: {src_img}")
            missing_imgs += 1

        # Copy label
        src_lbl = os.path.join(LABELS_FOLDER, stem + ".txt")
        dst_lbl = os.path.join(f"dataset/{split}/labels", stem + ".txt")
        shutil.copy(src_lbl, dst_lbl)

    print(f"{split} copied — missing images: {missing_imgs}")

copy_split(train_stems, "train")
copy_split(val_stems, "val")

print("\nDataset split complete!")
print(f"dataset/train/images → {len(os.listdir('dataset/train/images'))} files")
print(f"dataset/train/labels → {len(os.listdir('dataset/train/labels'))} files")
print(f"dataset/val/images   → {len(os.listdir('dataset/val/images'))} files")
print(f"dataset/val/labels   → {len(os.listdir('dataset/val/labels'))} files")

Total labeled frames: 96
Train: 85 | Val: 11
train copied — missing images: 0
val copied — missing images: 0

Dataset split complete!
dataset/train/images → 85 files
dataset/train/labels → 85 files
dataset/val/images   → 11 files
dataset/val/labels   → 11 files


## YOLOv8x-Pose Fine-Tuning

Fine-tunes the **YOLOv8x-Pose** (Extra Large) model on the custom bowling keypoint dataset with a real-time epoch progress tracker.

### Training Configuration
| Parameter | Value |
|-----------|-------|
| Model | YOLOv8x-Pose (pretrained) |
| Epochs | 100 (early stopping patience=20) |
| Image Size | 640px |
| Batch Size | 16 |
| Device | GPU (CUDA)/CPU |
| Flip Augmentation | Disabled (directional action) |

### Features
- Custom `on_train_epoch_end` callback with live loss and remaining epoch display
- Early stopping if no improvement for 20 consecutive epochs


In [14]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
from tqdm.notebook import tqdm
import time

model = YOLO('yolov8x-pose.pt')

EPOCHS = 100

# Custom callback to track progress
epoch_bar = tqdm(total=EPOCHS, desc="Training Progress", unit="epoch", colour="green")

current_epoch = [0]

def on_epoch_end(trainer):
    current_epoch[0] += 1
    ep = current_epoch[0]

    # Get latest metrics
    loss = trainer.loss.item() if hasattr(trainer.loss, 'item') else float(trainer.loss)

    epoch_bar.set_postfix({
        'epoch': f'{ep}/{EPOCHS}',
        'loss': f'{loss:.4f}',
        'remaining': f'{EPOCHS - ep} left'
    })
    epoch_bar.update(1)

model.add_callback('on_train_epoch_end', on_epoch_end)

results = model.train(
    data='bowling_pose.yaml',
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    patience=20,
    augment=True,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.5,
    project='bowling_runs',
    name='exp1',
    save=True,
    plots=True,
    device=0
)

epoch_bar.close()
print("Training complete!")
print("Best model saved at: bowling_runs/exp1/weights/best.pt")

## Training Log Viewer

Reads the YOLO training `results.csv` and prints a formatted epoch-by-epoch training log directly in the notebook output.

### What It Shows
| Column | Description |
|--------|-------------|
| BoxLoss / PoseLoss | Training losses per epoch |
| PosemAP50 | Pose keypoint accuracy at IoU 0.50 |
| PosePrec / PoseRec | Precision and Recall for keypoints |
| ValBox / ValPose | Validation losses |

### Usage
Point `results.csv` to your training output folder and run best epoch is auto-marked with `← BEST` in the output.

In [7]:
import pandas as pd

df = pd.read_csv('graphs/results.csv')
df.columns = df.columns.str.strip()

print("=" * 100)
print(f"{'TRAINING LOG':^100}")
print("=" * 100)
print(f"{'Epoch':>6} {'BoxLoss':>9} {'PoseLoss':>10} {'ClsLoss':>9} {'DFLLoss':>9} | {'BoxmAP50':>9} {'PosemAP50':>10} {'PosePrec':>9} {'PoseRec':>9} | {'ValBox':>8} {'ValPose':>9}")
print("-" * 100)

for _, row in df.iterrows():
    epoch     = int(row['epoch'])
    tr_box    = row['train/box_loss']
    tr_pose   = row['train/pose_loss']
    tr_cls    = row['train/cls_loss']
    tr_dfl    = row['train/dfl_loss']
    map50_b   = row['metrics/mAP50(B)']
    map50_p   = row['metrics/mAP50(P)']
    prec_p    = row['metrics/precision(P)']
    rec_p     = row['metrics/recall(P)']
    val_box   = row['val/box_loss']
    val_pose  = row['val/pose_loss']

    # Mark best epoch
    marker = " ← BEST" if epoch == 55 else ""

    print(f"{epoch:>6} {tr_box:>9.4f} {tr_pose:>10.4f} {tr_cls:>9.4f} {tr_dfl:>9.4f} | {map50_b:>9.4f} {map50_p:>10.4f} {prec_p:>9.4f} {rec_p:>9.4f} | {val_box:>8.4f} {val_pose:>9.4f}{marker}")

print("=" * 100)
print(f"\nTotal Epochs    : {len(df)}")
print(f"Best Epoch      : 55")
best = df[df['epoch'] == 55].iloc[0]
print(f"Best PosemAP50  : {best['metrics/mAP50(P)']:.4f}")
print(f"Best PosemAP50-95: {best['metrics/mAP50-95(P)']:.4f}")
print(f"Final Train Loss: {df.iloc[-1]['train/box_loss']:.4f}")

                                            TRAINING LOG                                            
 Epoch   BoxLoss   PoseLoss   ClsLoss   DFLLoss |  BoxmAP50  PosemAP50  PosePrec   PoseRec |   ValBox   ValPose
----------------------------------------------------------------------------------------------------
     1    3.7873     6.1511   23.5278    3.0175 |    0.0000     0.0000    0.0000    0.0000 |   3.7455    5.1071
     2    3.3228     4.6079   13.4468    2.7494 |    0.2910     0.5615    0.5933    0.5454 |   1.8567    2.3691
     3    2.1325     2.0647    1.9058    1.6443 |    0.0069     0.0203    0.0245    0.3636 |   2.9529    2.9489
     4    1.6248     1.5767    2.1556    1.2876 |    0.0000     0.0000    0.0000    0.0000 |   0.0000    0.0000
     5    1.6520     1.6113    1.1540    1.2693 |    0.0000     0.0000    0.0000    0.0000 |      nan       nan
     6    1.6078     0.9839    1.2102    1.2372 |    0.0000     0.0000    0.0000    0.0000 |      nan   12.0000
     7    1.80

## Run the inference Video

In [12]:
from ultralytics import YOLO

model = YOLO("weights/best.pt")

results = model.predict(source="clip.mp4", show=True, save=True)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/517) d:\Cricket\clip.mp4: 384x640 1 bowler, 1188.2ms
video 1/1 (frame 2/517) d:\Cricket\clip.mp4: 384x640 1 bowler, 1210.7ms
video 1/1 (frame 3/517) d:\Cricket\clip.mp4: 384x640 1 bowler, 1121.8ms
video 1/1 (frame 4/517) d:\Cricket\clip.mp4: 384x640 1 bowler, 1196.8ms
video 1/1 (frame 5/517) d:\Cricket\clip.mp4: 384x640 1 bowler, 1207.3ms
video 1/1 (frame 6/517) d:\Cricket\clip.mp4: 384x640 1 bowler, 1016.1ms
video 1/1 (frame 7/517) d:\Cr

# Bowling Biomechanics Inference Engine

Runs the fine-tuned **YOLOv8x-Pose** model on a cricket bowling video and generates a fully annotated output video with live biomechanical overlays and analytics graphs.

---

## What the Output Looks Like
```
┌─────────────────────────────────┬──────────────────┐
│                                 │  BOWLING ANALYSIS│
│   Original Video Frame          │                  │
│   + Keypoints (S, E, W)         │  [Elbow Angle]   │
│   + Arm Lines                   │   Graph (cyan)   │
│   + Wrist Trail (gradient)      │                  │
│   + Fan Lines from Elbow        │  [Wrist Speed]   │
│   + Elbow Angle Text            │   Graph (green)  │
│                                 │                  │
└─────────────────────────────────┴──────────────────┘
         Original Width                 420px
```

---

## Configuration Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `CONF_THRESHOLD` | 0.3 | Minimum keypoint confidence to accept |
| `TRAIL_LENGTH` | 300 | How many past wrist positions to remember |
| `FAN_INTERVAL` | 25 | Draw a fan line every 25th trail point |
| `MAX_JUMP` | 100 | Reject wrist jump > 100px between frames |
| `SMOOTH_WINDOW` | 20 | Average last 20 trail points for clean arc |
| `ARM_LENGTH_M` | 2.5 | Real-world arm length reference for m/s conversion |

---

## Step-by-Step Logic

### Step 1 - Read Video
```
cap = cv2.VideoCapture("clip.mp4")
```
Reads video metadata — FPS, width, height, total frames. Creates output writer with original width + 420px panel.

---

### Step 2 - Run YOLO on Each Frame
```
results = model(frame, verbose=False)
```
For each frame, YOLO returns 3 keypoint predictions with (x, y, confidence):
- **Index 0** → Shoulder (blue dot, label `S`)
- **Index 1** → Elbow (orange dot, label `E`)
- **Index 2** → Wrist (green dot, label `W`)

Any keypoint below `CONF_THRESHOLD = 0.3` is ignored.

---

### Step 3 - Jump Filter
```
if distance(new_wrist, prev_wrist) < MAX_JUMP:
    accept point
else:
    reject point
```
Fast bowling creates sudden wrist movements. If the wrist moves more than **100 pixels** between two consecutive frames, the point is rejected as noise. This keeps the trail clean and smooth.

---

### Step 4 - Wrist Trail
```
wrist_trail = deque(maxlen=300)
```
Stores the last 300 accepted wrist positions. A **20-frame moving average** is applied before drawing to smooth out any remaining jitter. The trail is drawn with a **red → orange → yellow gradient** that grows thicker toward the current frame. A white dot is placed every 8 points to mark intervals.

---

### Step 5 - Fan Lines
```
sampled = smoothed[::25]   # every 25th point
cv2.line(frame, elbow_pt, wp, fan_color, 1)
```
Lines are drawn from the **current elbow position** to every 25th point in the wrist trail. Color fades from **cyan → white** as the trail gets more recent. Fan lines within 500px of elbow only — distant points are skipped to avoid noisy lines.

---

### Step 6 - Elbow Angle Calculation
```
v1 = shoulder → elbow vector
v2 = wrist   → elbow vector
angle = arccos(dot(v1, v2) / (|v1| × |v2|))
```
Uses the **dot product formula** to calculate the angle at the elbow joint every frame. Displayed as cyan text next to the elbow point. Plotted live on the top graph (0–180°).

---

### Step 7 - Wrist Speed in m/s
```
speed_px  = distance(wrist_now, wrist_prev) × fps
px_to_m   = ARM_LENGTH_M / arm_length_in_pixels
speed_ms  = speed_px × px_to_m
```
**Three-step conversion:**
1. Pixel displacement per frame × FPS = pixels per second
2. Measure arm length in pixels from shoulder to wrist
3. Divide real arm length (2.5m) by pixel arm length = scale factor
4. Multiply to get real-world m/s

A **7-frame rolling average** is applied to the speed buffer to prevent spiky graph values.

---

### Step 8 - Side Panel Graphs
Two live graphs are drawn on the right panel every frame:

| Graph | Color | Y-axis Range |
|-------|-------|-------------|
| Elbow Angle | Cyan | 0° - 180° |
| Wrist Speed | Green | 0 - 20 m/s |

Each graph has a semi-transparent fill under the curve, grid lines, y-axis labels, and a live current-value display in the top right corner.

---

### Step 9 - Combine and Save
```
combined = np.hstack([frame, panel])
out.write(combined)
```
Original annotated frame and analytics panel are horizontally stacked and written to `output_analysis.mp4` frame by frame.

---




In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
from collections import deque
from tqdm.notebook import tqdm
import math

# ---- CONFIG ----
MODEL_PATH = "weights/best.pt"
VIDEO_PATH = "clip.mp4"
OUTPUT_PATH = "output_analysis.mp4"
CONF_THRESHOLD = 0.3
TRAIL_LENGTH = 300
FAN_INTERVAL = 25
MAX_JUMP = 100
SMOOTH_WINDOW = 20
ARM_LENGTH_M = 2.5    
# ----------------

model = YOLO(MODEL_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

GRAPH_W = 420
OUT_W = w + GRAPH_W

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (OUT_W, h))

wrist_trail   = deque(maxlen=TRAIL_LENGTH)
angle_history = deque(maxlen=200)
speed_history = deque(maxlen=200)
speed_buffer  = deque(maxlen=7)    

prev_wrist = None
frame_count = 0
detected_frames = 0

def is_valid_jump(new_pt, prev_pt, max_jump):
    if prev_pt is None:
        return True
    dist = math.sqrt((new_pt[0]-prev_pt[0])**2 + (new_pt[1]-prev_pt[1])**2)
    return dist < max_jump

def smooth_trail(trail, window=20):
    pts = list(trail)
    if len(pts) < window:
        return pts
    smoothed = []
    for i in range(len(pts)):
        start = max(0, i - window // 2)
        end   = min(len(pts), i + window // 2 + 1)
        wp = pts[start:end]
        avg_x = int(sum(p[0] for p in wp) / len(wp))
        avg_y = int(sum(p[1] for p in wp) / len(wp))
        smoothed.append((avg_x, avg_y))
    return smoothed

def get_trail_color(i, total):
    ratio = i / max(total, 1)
    if ratio < 0.5:
        r, g, b = 255, int(ratio * 2 * 140), 0
    else:
        r, g, b = 255, int(140 + (ratio-0.5) * 2 * 115), 0
    return (b, g, r)

def get_fan_color(i, total):
    ratio = i / max(total, 1)
    val = int(150 + ratio * 105)
    return (val, val, 255)

def calc_angle(shoulder, elbow, wrist):
    v1 = (shoulder[0]-elbow[0], shoulder[1]-elbow[1])
    v2 = (wrist[0]-elbow[0],   wrist[1]-elbow[1])
    dot  = v1[0]*v2[0] + v1[1]*v2[1]
    mag1 = math.sqrt(v1[0]**2 + v1[1]**2)
    mag2 = math.sqrt(v2[0]**2 + v2[1]**2)
    if mag1 * mag2 == 0:
        return 0
    return math.degrees(math.acos(max(-1, min(1, dot/(mag1*mag2)))))

def draw_graph(panel, history, title, unit, color, y_offset, max_val, graph_h=160):
    gx, gy = 25, y_offset
    gw = GRAPH_W - 50
    gh = graph_h

    cv2.rectangle(panel, (gx-8, gy-35), (gx+gw+8, gy+gh+15), (25,25,25), -1)
    cv2.rectangle(panel, (gx-8, gy-35), (gx+gw+8, gy+gh+15), (80,80,80), 1)
    cv2.putText(panel, title, (gx, gy-15),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (220,220,220), 2)

    for i in range(5):
        y_grid = gy + int(gh * i / 4)
        cv2.line(panel, (gx, y_grid), (gx+gw, y_grid), (45,45,45), 1)
        val_label = int(max_val - max_val * i / 4)
        cv2.putText(panel, str(val_label), (gx-5, y_grid+4),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.35, (100,100,100), 1)

    pts = list(history)
    if len(pts) > 1:
        poly_pts = []
        for i, val in enumerate(pts):
            if val is None:
                continue
            x = gx + int(i / len(pts) * gw)
            y = gy + gh - int(min(val, max_val) / max_val * gh)
            poly_pts.append((x, y))

        if len(poly_pts) > 1:
            fill_pts = [(poly_pts[0][0], gy+gh)] + poly_pts + [(poly_pts[-1][0], gy+gh)]
            fill_arr = np.array(fill_pts, dtype=np.int32)
            overlay = panel.copy()
            cv2.fillPoly(overlay, [fill_arr], color)
            cv2.addWeighted(overlay, 0.2, panel, 0.8, 0, panel)
            for i in range(1, len(poly_pts)):
                cv2.line(panel, poly_pts[i-1], poly_pts[i], color, 2)

    if pts and pts[-1] is not None:
        cv2.putText(panel, f"{pts[-1]:.1f} {unit}",
                   (gx+gw-120, gy-15),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

progress = tqdm(total=total_frames, desc="Inference", unit="frame", colour="green")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    panel = np.zeros((h, GRAPH_W, 3), dtype=np.uint8)
    panel[:] = (15, 15, 15)

    results = model(frame, verbose=False)
    detected = False

    shoulder_pt = elbow_pt = wrist_pt = None

    for result in results:
        if result.keypoints is not None and len(result.keypoints.data) > 0:
            kps = result.keypoints.data[0]

            pts = {}
            for kp_idx in range(3):
                if kp_idx < len(kps):
                    x, y, conf = kps[kp_idx]
                    x, y, conf = int(x.item()), int(y.item()), conf.item()
                    if conf > CONF_THRESHOLD and x > 0 and y > 0:
                        pts[kp_idx] = (x, y)

            if pts:
                detected = True
                shoulder_pt = pts.get(0)
                elbow_pt    = pts.get(1)
                wrist_pt    = pts.get(2)

                if wrist_pt:
                    last = wrist_trail[-1] if wrist_trail else None
                    if is_valid_jump(wrist_pt, last, MAX_JUMP):
                        wrist_trail.append(wrist_pt)

                if shoulder_pt and elbow_pt:
                    cv2.line(frame, shoulder_pt, elbow_pt, (255,255,255), 2)
                if elbow_pt and wrist_pt:
                    cv2.line(frame, elbow_pt, wrist_pt, (255,255,255), 2)

                for kp_idx, pt in pts.items():
                    colors = {0:(100,100,255), 1:(0,165,255), 2:(0,255,100)}
                    names  = {0:'S', 1:'E', 2:'W'}
                    cv2.circle(frame, pt, 9, colors[kp_idx], -1)
                    cv2.circle(frame, pt, 11, (255,255,255), 1)
                    cv2.putText(frame, names[kp_idx], (pt[0]+12, pt[1]-10),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.55, colors[kp_idx], 2)

                if shoulder_pt and elbow_pt and wrist_pt:
                    angle = calc_angle(shoulder_pt, elbow_pt, wrist_pt)
                    angle_history.append(angle)
                    cv2.putText(frame, f"{angle:.1f}",
                               (elbow_pt[0]+15, elbow_pt[1]-15),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
                else:
                    angle_history.append(angle_history[-1] if angle_history else 0)

                # px to meter ratio
                if shoulder_pt and wrist_pt:
                    arm_px = math.sqrt((wrist_pt[0]-shoulder_pt[0])**2 +
                                       (wrist_pt[1]-shoulder_pt[1])**2)
                    px_to_m = ARM_LENGTH_M / arm_px if arm_px > 0 else 0.001
                else:
                    px_to_m = 0.001

                # Wrist speed in m/s with smoothing buffer
                if wrist_pt and prev_wrist:
                    if is_valid_jump(wrist_pt, prev_wrist, MAX_JUMP):
                        dx = wrist_pt[0] - prev_wrist[0]
                        dy = wrist_pt[1] - prev_wrist[1]
                        speed_px = math.sqrt(dx**2 + dy**2) * fps
                        speed_ms = speed_px * px_to_m
                        speed_buffer.append(speed_ms)
                        # Use rolling average for smooth display
                        smoothed_speed = sum(speed_buffer) / len(speed_buffer)
                        speed_history.append(smoothed_speed)
                    else:
                        speed_history.append(speed_history[-1] if speed_history else 0)
                else:
                    speed_history.append(speed_history[-1] if speed_history else 0)

                prev_wrist = wrist_pt

    # Smoothed wrist trail
    smoothed = smooth_trail(wrist_trail, SMOOTH_WINDOW)
    for i in range(1, len(smoothed)):
        color = get_trail_color(i, len(smoothed))
        thickness = max(2, int(6 * i / len(smoothed)))
        cv2.line(frame, smoothed[i-1], smoothed[i], color, thickness)
        if i % 8 == 0:
            cv2.circle(frame, smoothed[i], 4, (255,255,255), -1)

    # Fan lines
    if elbow_pt and len(smoothed) >= FAN_INTERVAL:
        sampled = smoothed[::FAN_INTERVAL]
        for i, wp in enumerate(sampled):
            dist = math.sqrt((wp[0]-elbow_pt[0])**2 + (wp[1]-elbow_pt[1])**2)
            if dist < 500:
                color = get_fan_color(i, len(sampled))
                cv2.line(frame, elbow_pt, wp, color, 1)

    # Panel
    cv2.putText(panel, "BOWLING ANALYSIS", (15, 35),
               cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0,200,255), 2)
    cv2.line(panel, (15, 45), (GRAPH_W-15, 45), (60,60,60), 1)

    draw_graph(panel, angle_history, "Elbow Angle", "deg",
              (0,255,255), 75, 180, graph_h=int(h//2 - 120))

    draw_graph(panel, speed_history, "Wrist Speed", "m/s",
              (0,255,100), h//2 + 20, 20, graph_h=int(h//2 - 120))

    cv2.putText(panel, f"Frame {frame_count}/{total_frames}",
               (15, h-30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (120,120,120), 1)

    combined = np.hstack([frame, panel])
    out.write(combined)

    if detected:
        detected_frames += 1

    frame_count += 1
    progress.set_postfix({
        'left': total_frames - frame_count,
        'detected': f'{detected_frames}/{frame_count}'
    })
    progress.update(1)

cap.release()
out.release()
progress.close()

print(f"\nDone!")
print(f"Detection rate: {detected_frames/frame_count*100:.1f}%")



Inference:   0%|          | 0/517 [00:00<?, ?frame/s]


Done!
Detection rate: 99.6%


---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
